In [1]:
import pandas as pd
import numpy as np

# Indlæs filen
df_landbrug = pd.read_csv('Data69/Kapital_landbrugsdata.csv')

#filter for løbende priser og foregående års priser
df_landbrug_loebende=df_landbrug.loc[df_landbrug['PRISENHED']=='Løbende priser']
df_landbrug_loebende=df_landbrug_loebende.loc[df_landbrug_loebende['TID']!=1966]
df_landbrug_for=df_landbrug.loc[df_landbrug['PRISENHED']=='Forige års priser']

#fjern prishenhed
df_landbrug_loebende.drop(columns=['PRISENHED'],inplace=True)
df_landbrug_for.drop(columns=['PRISENHED'],inplace=True)

#set index
df_landbrug_loebende.set_index(['BEHOLD','BRANCHE', 'TID'],inplace=True)
df_landbrug_for.set_index(['BEHOLD','BRANCHE', 'TID'],inplace=True)

#tjek
df_landbrug_loebende

C:\Users\b431385\AppData\Local\Temp\ipykernel_15296\1027462865.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_landbrug_for.drop(columns=['PRISENHED'],inplace=True)


INDHOLD
BEHOLD                                           BRANCHE                                    TID           
AN.11 Faste aktiver, nettobeholdning ultimo året 01000 Landbrug og gartneri                 1967   27567.0
                                                 10120 Føde-, drikke- og tobaksvareindustri 1967    4962.0
P.51g Faste bruttoinvesteringer                  01000 Landbrug og gartneri                 1967       NaN
                                                 10120 Føde-, drikke- og tobaksvareindustri 1967       NaN
AN.11 Faste aktiver, nettobeholdning ultimo året 01000 Landbrug og gartneri                 1968   29508.0
...                                                                                                    ...
P.51g Faste bruttoinvesteringer                  REST_ANVENDELSE Øvrige brancher            2018  472349.0
                                                                                            2019  476145.0
                                                                                            2020  501927.0
                                                                                            2021  575711.0
                                                                                            2022  636711.0

[342 rows x 1 columns]

In [2]:
# Beregn Pt/Pt-1
Pt_Pt=df_landbrug_loebende/df_landbrug_for

# Vi bruger .unstack() til at flytte 'TID' fra index til kolonner
df_wide = Pt_Pt['INDHOLD'].unstack(level='TID')

# Lav en tom tabel i samme form til vores resultat (Pt)
Pt = df_wide.copy()
Pt.loc[:, :] = np.nan  # Tøm tabellen for tal
Pt[2020] = 1.0         # Sæt 2020 til 1.0
# Find alle år i dit datasæt og sortér dem
years = sorted(df_wide.columns)

# FREMAD: Fra 2021 og op (Pt = Pt-1 * Pt_Pt_nu)
for y in years:
    if y > 2020:
        Pt[y] = Pt[y-1] * df_wide[y]

# BAGUD: Fra 2019 og ned (Pt = Pt+1 / Pt_Pt_næste_år)
for y in reversed(years):
    if y < 2020:
        Pt[y] = Pt[y+1] / df_wide[y+1]

df_Pt_final = Pt.stack(future_stack=True).reset_index()

# Navngiv kolonnerne som før
df_Pt_final.columns = ['BEHOLD','BRANCHE', 'TID', 'Pt']
df_Pt_final.to_csv('Data69/landbrugsdata_prisindeks_kapital.csv', index=False)
df_Pt_final

,BEHOLD,BRANCHE,TID,Pt
0,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1966,0.000000
1,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1967,0.122785
2,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1968,0.131517
3,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1969,0.143119
4,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1970,0.155033
...,...,...,...,...
349,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2020,1.000000
350,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2021,1.018521
351,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2022,1.104939
352,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2023,NaN


In [3]:
df_Pt_final.set_index(['BEHOLD','BRANCHE', 'TID'],inplace=True)
df_Xt=df_landbrug_loebende['INDHOLD']/df_Pt_final['Pt']
df_Xt.columns = ['BEHOLD', 'BRANCHE', 'TID', 'Xt']
df_Xt_final = df_Xt.to_frame(name='Xt')

# 2. Fjern MultiIndex så den bliver "flad"
df_Xt_final = df_Xt_final.reset_index()

df_Xt_final.to_csv('Data69/landbrugsdata_mængdeindeks_kapital.csv', index=False)
df_Xt_final

,BEHOLD,BRANCHE,TID,Xt
0,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1966,NaN
1,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1967,224514.512147
2,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1968,224366.456415
3,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1969,225176.761432
4,"AN.11 Faste aktiver, nettobeholdning ultimo året",01000 Landbrug og gartneri,1970,225616.927120
...,...,...,...,...
349,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2020,501927.000000
350,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2021,565242.000000
351,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2022,576240.780864
352,P.51g Faste bruttoinvesteringer,REST_ANVENDELSE Øvrige brancher,2023,NaN
